# Pydantic — Nível Básico

**Objetivo:** sair daqui sabendo criar modelos, ler erros de validação, restringir campos, usar os tipos prontos,
montar estruturas aninhadas e serializar para `dict`/JSON.

Teoria completa em [`pydantic.md`](pydantic.md). Roteiro:

1. Primeiro modelo: validação + conversão
2. Lendo um `ValidationError`
3. Obrigatório × opcional × default (e a pegadinha do `Optional`)
4. `Field`: restrições e metadados
5. Tipos prontos (`EmailStr`, `HttpUrl`, `SecretStr`, `UUID`, `Decimal`, `Literal`, datas)
6. Modelos aninhados
7. Entrada: `model_validate` e `model_validate_json`
8. Saída: `model_dump`, `model_dump_json`, `include`/`exclude`
9. Coerções que surpreendem
10. Resumo

In [1]:
import pydantic, sys
print(f"Python: {sys.version.split()[0]}")
print(f"Pydantic: {pydantic.VERSION}")

Python: 3.13.18
Pydantic: 2.13.5


## 1. Primeiro modelo: validação + conversão

Um modelo é uma classe que herda de `BaseModel`. Cada atributo anotado vira um **campo** validado.
Repare no `price="40"` — uma *string* — que vira `float` automaticamente (modo *lax*).

In [2]:
from pydantic import BaseModel, ConfigDict

class Product(BaseModel):
    name: str
    price: float
    in_stock: bool = True

keyboard = Product(name="Kumara", price=300)
mousepad = Product(name="Mousepad", price="40", in_stock="no")   # strings convertidas

print(f"keyboard: {keyboard}")
print(f"mousepad: {mousepad}")
print(f"type(mousepad.price): {type(mousepad.price)}")
print(f"type(mousepad.in_stock): {type(mousepad.in_stock)}")

keyboard: name='Kumara' price=300.0 in_stock=True
mousepad: name='Mousepad' price=40.0 in_stock=False
type(mousepad.price): <class 'float'>
type(mousepad.in_stock): <class 'bool'>


Diferente de uma `dataclass`, aqui o tipo **não é decorativo**: se não der para converter, o objeto nem é criado.

In [3]:
from pydantic import ValidationError

try:
    Product(name="Monitor", price="caro")
except ValidationError as e:
    print(f"{type(e).__name__}: {e}")

ValidationError: 1 validation error for Product
price
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='caro', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/float_parsing


## 2. Lendo um `ValidationError`

O erro é estruturado. `e.errors()` devolve uma lista de dicionários — perfeito para transformar em resposta de API ou log.
Cada item tem:

| chave | significado |
|---|---|
| `type` | código estável do erro (`int_parsing`, `missing`, `greater_than`...) — use este para lógica |
| `loc` | caminho do campo (tupla; em modelos aninhados vira `('items', 0, 'price')`) |
| `msg` | mensagem legível |
| `input` | valor recebido |
| `ctx` | contexto extra (ex.: o limite violado) |

O Pydantic **coleta todos os erros** de uma vez — não para no primeiro.

In [4]:
try:
    Product(name=123, price=None)
except ValidationError as e:
    print(f"quantidade de erros: {e.error_count()}")
    for err in e.errors(include_url=False):
        print(f"err: {err}")

quantidade de erros: 2
err: {'type': 'string_type', 'loc': ('name',), 'msg': 'Input should be a valid string', 'input': 123}
err: {'type': 'float_type', 'loc': ('price',), 'msg': 'Input should be a valid number', 'input': None}


## 3. Obrigatório × opcional × default

Regra da v2: **um campo é obrigatório se não tem default**, independente de aceitar `None`.

| declaração | obrigatório? | aceita `None`? |
|---|---|---|
| `x: int` | sim | não |
| `x: int = 0` | não | não |
| `x: int \| None` | **sim** | sim |
| `x: int \| None = None` | não | sim |

In [5]:
class Example(BaseModel):
    required: int
    with_default: int = 0
    nullable_but_required: int | None
    optional: int | None = None

try:
    Example(required=1)
except ValidationError as e:
    print(f'{e.errors(include_url=False)[0]["loc"]} -> {e.errors(include_url=False)[0]["type"]}')

print(f"Example(required=1, nullable_but_required=None): {Example(required=1, nullable_but_required=None)}")

('nullable_but_required',) -> missing
Example(required=1, nullable_but_required=None): required=1 with_default=0 nullable_but_required=None optional=None


### Defaults mutáveis
No Pydantic, `list = []` é seguro: cada instância recebe **uma cópia** do default. Ainda assim, `Field(default_factory=list)`
deixa a intenção explícita e é o único jeito de gerar defaults dinâmicos (ex.: `datetime.now`).

In [6]:
from datetime import datetime, timezone
from pydantic import Field

class Cart(BaseModel):
    items: list[str] = []
    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))

a, b = Cart(), Cart()
a.items.append("mouse")
print(f"a.items: {a.items}")
print(f"b.items: {b.items}")
print(f"listas diferentes: {a.items is not b.items}")

a.items: ['mouse']
b.items: []
listas diferentes: True


## 4. `Field`: restrições e metadados

`Field` declara regras **sem escrever código de validação**. As restrições mais usadas:

| tipo | restrições |
|---|---|
| números | `gt`, `ge`, `lt`, `le`, `multiple_of` |
| strings | `min_length`, `max_length`, `pattern` |
| listas/sets | `min_length`, `max_length` |
| `Decimal` | `max_digits`, `decimal_places` |

Metadados (`description`, `examples`, `title`) não validam nada, mas aparecem no **JSON Schema** — e isso vira documentação
de API e instrução para LLMs (veja `AI_use_cases`).

In [7]:
class Employee(BaseModel):
    name: str = Field(min_length=2, max_length=80, description="Nome completo")
    age: int = Field(ge=16, le=100)
    registration: str = Field(pattern=r"^EMP-\d{4}$", examples=["EMP-0042"])
    salary: float = Field(gt=0)
    skills: list[str] = Field(default_factory=list, max_length=10)

print(f'Employee(name="Ana", age=31, registration="EMP-0042", salary=8500): {Employee(name="Ana", age=31, registration="EMP-0042", salary=8500)}')

try:
    Employee(name="A", age=12, registration="42", salary=-1)
except ValidationError as e:
    for err in e.errors(include_url=False):
        print(f"{err['loc'][0]:<13} {err['type']:<22} {err['msg']}")

Employee(name="Ana", age=31, registration="EMP-0042", salary=8500): name='Ana' age=31 registration='EMP-0042' salary=8500.0 skills=[]
name          string_too_short       String should have at least 2 characters
age           greater_than_equal     Input should be greater than or equal to 16
registration  string_pattern_mismatch String should match pattern '^EMP-\d{4}$'
salary        greater_than           Input should be greater than 0


A forma moderna e **reutilizável** é `Annotated`: a restrição fica no *tipo*, que você pode importar e reaproveitar em vários modelos.

In [8]:
from typing import Annotated

PositiveMoney = Annotated[float, Field(gt=0, description="Valor monetário positivo")]
ShortText = Annotated[str, Field(min_length=1, max_length=120)]

class Invoice(BaseModel):
    title: ShortText
    total: PositiveMoney

class Refund(BaseModel):
    reason: ShortText
    amount: PositiveMoney

print(f'Invoice(title="Consultoria", total=1500): {Invoice(title="Consultoria", total=1500)}')

Invoice(title="Consultoria", total=1500): title='Consultoria' total=1500.0


## 5. Tipos prontos

Pydantic traz validadores para formatos do dia a dia. Os mais úteis em APIs e configs:

In [9]:
from decimal import Decimal
from typing import Literal
from uuid import UUID, uuid4
from datetime import date
from pydantic import EmailStr, HttpUrl, SecretStr

class Account(BaseModel):
    id: UUID = Field(default_factory=uuid4)
    email: EmailStr                         # requer: pip install "pydantic[email]"
    website: HttpUrl | None = None
    password: SecretStr                     # escondido em repr/print/log
    plan: Literal["free", "pro", "enterprise"] = "free"
    balance: Decimal = Decimal("0.00")      # Decimal para dinheiro, nunca float
    birthday: date | None = None

acc = Account(
    email="dante@example.com",
    website="https://example.com/perfil",
    password="s3nh4-sup3r",
    plan="pro",
    balance="10.50",
    birthday="1990-05-17",
)
print(f"acc: {acc}")
print(f"senha real: {acc.password.get_secret_value()}")
print(f"host do site: {acc.website.host}")

acc: id=UUID('4900e8e6-9a0b-4949-bb1d-4629fb738bcd') email='dante@example.com' website=HttpUrl('https://example.com/perfil') password=SecretStr('**********') plan='pro' balance=Decimal('10.50') birthday=datetime.date(1990, 5, 17)
senha real: s3nh4-sup3r
host do site: example.com


In [10]:
try:
    Account(email="nao-e-email", password="x", plan="gold", website="ftp//errado")
except ValidationError as e:
    for err in e.errors(include_url=False):
        print(f"{err['loc'][0]:<8} {err['type']:<18} {err['msg']}")

email    value_error        value is not a valid email address: An email address must have an @-sign.
website  url_parsing        Input should be a valid URL, relative URL without a base
plan     literal_error      Input should be 'free', 'pro' or 'enterprise'


## 6. Modelos aninhados

Um campo pode ser outro modelo, ou uma lista/dict de modelos. O Pydantic converte `dict` → objeto em qualquer profundidade,
e o `loc` do erro aponta o caminho exato.

In [11]:
class Customer(BaseModel):
    name: str
    email: EmailStr

class OrderItem(BaseModel):
    product: Product
    quantity: int = Field(ge=1)

class Order(BaseModel):
    customer: Customer
    items: list[OrderItem]

    def total(self) -> float:
        return sum(i.product.price * i.quantity for i in self.items)

payload = {
    "customer": {"name": "Guilherme", "email": "gui@example.com"},
    "items": [
        {"product": {"name": "Kumara", "price": 300}, "quantity": 1},
        {"product": {"name": "Mousepad", "price": "40"}, "quantity": "3"},
    ],
}
order = Order(**payload)
print(f"type(order.items[1].product).__name__: {type(order.items[1].product).__name__}")
print(f"total: {order.total()}")

type(order.items[1].product).__name__: Product
total: 420.0


In [12]:
bad = {
    "customer": {"name": "Guilherme", "email": "gui@"},
    "items": [{"product": {"name": "Kumara", "price": 300}, "quantity": 0}],
}
try:
    Order(**bad)
except ValidationError as e:
    for err in e.errors(include_url=False):
        print(f'{err["loc"]} -> {err["msg"]}')

('customer', 'email') -> value is not a valid email address: There must be something after the @-sign.
('items', 0, 'quantity') -> Input should be greater than or equal to 1


## 7. Entrada: `model_validate` e `model_validate_json`

| método | entrada | quando usar |
|---|---|---|
| `Model(**kwargs)` | argumentos nomeados | código Python |
| `Model.model_validate(obj)` | `dict` (ou objeto, com `from_attributes`) | dict vindo de outra lib |
| `Model.model_validate_json(raw)` | `str` / `bytes` JSON | corpo HTTP, arquivo, mensagem de fila, **saída de LLM** |

`model_validate_json` faz o parse do JSON dentro do Rust — é mais rápido e mais rigoroso do que `json.loads` + `model_validate`.

In [13]:
raw = '{"customer": {"name": "Ana", "email": "ana@example.com"}, "items": [{"product": {"name": "Cabo", "price": 19.9}, "quantity": 2}]}'

o1 = Order.model_validate_json(raw)
import json
o2 = Order.model_validate(json.loads(raw))
print(f"o1 == o2: {o1 == o2}")
print(f"total: {o1.total()}")

try:
    Order.model_validate_json('{"customer": {"name": "Ana"')   # JSON quebrado
except ValidationError as e:
    print(f'e.errors(include_url=False)[0]["type"]: {e.errors(include_url=False)[0]["type"]}')

o1 == o2: True
total: 39.8
e.errors(include_url=False)[0]["type"]: json_invalid


## 8. Saída: `model_dump` e `model_dump_json`

* `model_dump()` → `dict` com tipos Python (`datetime`, `UUID`, `Decimal` continuam objetos).
* `model_dump(mode="json")` → `dict` só com tipos compatíveis com JSON.
* `model_dump_json()` → `str` JSON pronto.

Filtros úteis: `include`, `exclude`, `exclude_none`, `exclude_defaults`, `exclude_unset` (este último é a base do PATCH, no intermediário).

In [14]:
print(f"acc.model_dump(): {acc.model_dump()}")
print()
print(f'acc.model_dump(mode="json"): {acc.model_dump(mode="json")}')

acc.model_dump(): {'id': UUID('4900e8e6-9a0b-4949-bb1d-4629fb738bcd'), 'email': 'dante@example.com', 'website': HttpUrl('https://example.com/perfil'), 'password': SecretStr('**********'), 'plan': 'pro', 'balance': Decimal('10.50'), 'birthday': datetime.date(1990, 5, 17)}

acc.model_dump(mode="json"): {'id': '4900e8e6-9a0b-4949-bb1d-4629fb738bcd', 'email': 'dante@example.com', 'website': 'https://example.com/perfil', 'password': '**********', 'plan': 'pro', 'balance': '10.50', 'birthday': '1990-05-17'}


In [15]:
print(f'acc.model_dump_json(indent=2, exclude={{"password", "id"}}):\n{acc.model_dump_json(indent=2, exclude={"password", "id"})}')
print(f'order.model_dump(include={{"customer": {{"name"}}, "items": {{0: {{"quantity"}}}}}}): {order.model_dump(include={"customer": {"name"}, "items": {0: {"quantity"}}})}')
minimal = Account(email="x@y.com", password="p")
print(f'minimal.model_dump(exclude_none=True, exclude={{"id", "password"}}): {minimal.model_dump(exclude_none=True, exclude={"id", "password"})}')

acc.model_dump_json(indent=2, exclude={"password", "id"}):
{
  "email": "dante@example.com",
  "website": "https://example.com/perfil",
  "plan": "pro",
  "balance": "10.50",
  "birthday": "1990-05-17"
}
order.model_dump(include={"customer": {"name"}, "items": {0: {"quantity"}}}): {'customer': {'name': 'Guilherme'}, 'items': [{'quantity': 1}]}
minimal.model_dump(exclude_none=True, exclude={"id", "password"}): {'email': 'x@y.com', 'plan': 'free', 'balance': Decimal('0.00')}


## 9. Coerções que surpreendem

O modo lax converte bastante coisa, mas **não converte quando perderia informação**. Conheça o comportamento antes de confiar nele:

In [16]:
from pydantic import TypeAdapter

casos = [
    (int, "42"), (int, 42.0), (int, 42.5), (int, "4_2"),
    (float, "1e3"), (bool, "yes"), (bool, "0"), (bool, "talvez"),
    (str, 42), (date, "2026-11-10"), (date, 1_700_000_000),
]
for tipo, valor in casos:
    try:
        r = repr(TypeAdapter(tipo).validate_python(valor))
    except ValidationError as e:
        r = "ERRO: " + e.errors()[0]["type"]
    print(f"{tipo.__name__:<6} {valor!r:<14} -> {r}")

int    '42'           -> 42
int    42.0           -> 42
int    42.5           -> ERRO: int_from_float
int    '4_2'          -> 42
float  '1e3'          -> 1000.0
bool   'yes'          -> True
bool   '0'            -> False
bool   'talvez'       -> ERRO: bool_parsing
str    42             -> ERRO: string_type
date   '2026-11-10'   -> datetime.date(2026, 11, 10)
date   1700000000     -> ERRO: date_from_datetime_inexact


Observações que costumam pegar gente experiente:

* `"4_2"` vira `42` — o Pydantic aceita `_` como separador, igual ao literal Python `4_2`. Num campo de CEP/telefone isso seria um bug silencioso: use `str` com `pattern`.
* `42.5 → int` falha (`int_from_float`): o lax nunca descarta informação.
* `42 → str` falha: a v2 não converte número para string (a v1 convertia). Se precisar, `ConfigDict(coerce_numbers_to_str=True)`.
* Timestamp → `date` só funciona se cair exatamente à meia-noite UTC; caso contrário `date_from_datetime_inexact`.

Como ficam as correções dos dois primeiros casos:

In [17]:
# Problema: CEP como int aceita "0_1310100" e perde o zero à esquerda
class AddressWrong(BaseModel):
    cep: int

print(f'AddressWrong(cep="0_1310100").cep: {AddressWrong(cep="0_1310100").cep}  <- zero perdido, "_" aceito')

# Correção: identificador numérico é str com pattern (nunca int)
class Address(BaseModel):
    cep: str = Field(pattern=r"^\d{8}$")

print(f'Address(cep="01310100").cep: {Address(cep="01310100").cep}')
for bad in ("0_1310100", "1310100", 1310100):
    try:
        Address(cep=bad)
    except ValidationError as e:
        print(f"Address(cep={bad!r}): {e.errors()[0]['type']}")

AddressWrong(cep="0_1310100").cep: 1310100  <- zero perdido, "_" aceito
Address(cep="01310100").cep: 01310100
Address(cep='0_1310100'): string_pattern_mismatch
Address(cep='1310100'): string_pattern_mismatch
Address(cep=1310100): string_type


In [18]:
# Problema: campo str recebendo número de um sistema legado
class LegacyRecord(BaseModel):
    code: str

try:
    LegacyRecord(code=1234)
except ValidationError as e:
    print(f"LegacyRecord(code=1234): {e.errors()[0]['type']}")

# Correção (quando a conversão é desejada): coerce_numbers_to_str
class LegacyRecordFixed(BaseModel):
    model_config = ConfigDict(coerce_numbers_to_str=True)
    code: str

print(f"LegacyRecordFixed(code=1234).code: {LegacyRecordFixed(code=1234).code!r}")

LegacyRecord(code=1234): string_type
LegacyRecordFixed(code=1234).code: '1234'


Quando a conversão silenciosa é perigosa (ex.: dinheiro, IDs, flags de segurança), ligue o **modo strict** — por campo, por modelo ou por chamada:

In [19]:
class Transfer(BaseModel):
    model_config = ConfigDict(strict=True)
    amount_cents: int
    confirmed: bool

print(f"Transfer(amount_cents=1000, confirmed=True): {Transfer(amount_cents=1000, confirmed=True)}")
try:
    Transfer(amount_cents="1000", confirmed="yes")
except ValidationError as e:
    print(f'[err["type"] for err in e.errors()]: {[err["type"] for err in e.errors()]}')

Transfer(amount_cents=1000, confirmed=True): amount_cents=1000 confirmed=True
[err["type"] for err in e.errors()]: ['int_type', 'bool_type']


## 10. Resumo

| conceito | lembrete |
|---|---|
| `BaseModel` | tipos são **contrato**, validados em runtime |
| erros | `ValidationError.errors()` → lista estruturada (`type`, `loc`, `msg`) |
| opcional | só é opcional se tiver **default** |
| `Field` / `Annotated` | restrições declarativas e reutilizáveis + metadados para o JSON Schema |
| tipos prontos | `EmailStr`, `HttpUrl`, `SecretStr`, `UUID`, `Decimal`, `Literal`, datas |
| entrada | `model_validate` (dict) · `model_validate_json` (str/bytes) |
| saída | `model_dump()` (Python) · `model_dump(mode="json")` · `model_dump_json()` |
| lax × strict | lax converte o que for seguro; strict não converte nada |

### Exercícios
1. Crie um modelo `Address` (CEP com `pattern` de 8 dígitos, UF como `Literal` de 3 estados) e aninhe em `Customer`.
2. Faça um `Order` com 2 erros em itens diferentes e imprima só os `loc` dos erros.
3. Serialize um `Account` para JSON sem `password` e sem campos `None`.

➡️ Próximo: [`02_pydantic_intermediario.ipynb`](02_pydantic_intermediario.ipynb)